# Load GRB2 with epik-dev (`model.load`)

This notebook loads the GRB2 dataset, constructs an `epik_dev` model, and restores parameters with:

- `model.set_data(X, y, y_var)`
- `model.load(path)`

In [1]:
import sys
from pathlib import Path

import numpy as np
import pandas as pd
import torch

sys.path.insert(0, "./modules/")

import epik_loader
from dataset_loader import get_dataset_spec


In [2]:
# Use epik-dev explicitly
epik_loader.ensure_epik_loaded(epik_flavor="epik_dev")

from epik.kernel import ConnectednessKernel
from epik.model import EpiK


In [ ]:
dataset = "GRB2"
device = "cuda" if torch.cuda.is_available() else "cpu"

spec = get_dataset_spec(dataset)

data = pd.read_csv(spec.data_csv, index_col=spec.index_col)
X = data.index.to_numpy()  # sequence strings for epik-dev
y = data[spec.y_column].to_numpy(dtype=float) + float(spec.phenotype_offset)
y_var = np.zeros_like(y)

alpha_df = pd.read_csv(spec.alphabet_csv, index_col=0)
alphabet_list = [[str(r["0"]), str(r["1"])] for _, r in alpha_df.iterrows()]

kernel = ConnectednessKernel(alphabet_list=alphabet_list) # can just do "Connectedness"
model = EpiK(kernel, alphabet_list=alphabet_list, train_noise=True, train_mean = True,device=device)

path = str(spec.model_params)

# Requested loading commands
model.set_data(X, y, y_var)
model.load(path)

print("Loaded:", path)
print("n_samples:", len(X), "seq_length:", len(X[0]), "device:", device)


Loaded: /cluster/tufts/pettilab/spetti04/gp_related/fitness_landscape_data/grb2_models/grb2.Connectedness.58.params.pth
n_samples: 71233 seq_length: 34 device: cuda


/cluster/tufts/pettilab/spetti04/condaenv/epik/lib/python3.8/site-packages/gpytorch/likelihoods/noise_models.py:150: NumericalWarning: Very small noise values detected. This will likely lead to numerical instabilities. Rounding small noise values up to 0.0001.
  warnings.warn(
/cluster/tufts/pettilab/spetti04/gp_related/epik-dev/epik/model.py:683: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_